# Decrypt datasets

Here is where **attestation** takes place. We need the key to decrypt the datasets, but it is stored in a remote attester (Trustee), and it will be provided to us only if **attestation** is successful, meaning the software & hardware of the CVM have not been tampered with.

By doing this we ensure that the CVM is **safe** and having the right hardware and software running prevents any attacker from fetching the transactions while they are being read by the model (**data in use** security). This is possible because the hardware inside the CVM makes sure that all data being loaded in the memory is encrypted, so that if an attacker tries to do a physical/virtual memory dump, the output will only be encrypted/zeroed blobs of memory.

This is what **Confidential Computing** is about: securing data in use.

Let's start by fetching the required key (note here we won't use `https`):

In [ ]:
! wget http://127.0.0.1:8006/cdh/resource/default/fraud-detection/dataset_key -O dataset.key

Checking back in our Trustee running in the secure environment, we will see that attestation happened and the key was successfully sent to this notebook:

```
[2025-04-24T13:02:44Z INFO  actix_web::middleware::logger] 10.88.0.14 "POST /kbs/v0/auth HTTP/1.1" 200 74 "-" "attestation-agent-kbs-client/0.1.0" 0.000246
[2025-04-24T13:02:48Z INFO  attestation_service] AzTdxVtpm Verifier/endorsement check passed.
[2025-04-24T13:02:48Z INFO  actix_web::middleware::logger] 10.88.0.14 "POST /kbs/v0/attest HTTP/1.1" 200 6384 "-" "attestation-agent-kbs-client/0.1.0" 1.211521
[2025-04-24T13:02:49Z INFO  actix_web::middleware::logger] 10.88.0.15 "GET /kbs/v0/resource/default/fraud-detection/dataset_key HTTP/1.1" 200 530 "-" "attestation-agent-kbs-client/0.1.0" 0.001043
```

Since we got a key, let's now decrypt the models!

In [ ]:
%%bash

KEY_FILE=dataset.key
DATASET_SRC=downloaded_datasets
DATASET_DEST=datasets_dec

mkdir -p $DATASET_DEST
rm -rf $DATASET_DEST/*

for file in $DATASET_SRC/*; do
    fname=$(basename $file)
    fname=${fname%.enc}
    openssl enc -d -aes-256-cfb -pbkdf2 -kfile $KEY_FILE -in $file -out $DATASET_DEST/$fname
    echo "Decrypted" $DATASET_DEST/$fname":"
    head -n 5 $DATASET_DEST/$fname
    echo ""
done

ls $DATASET_DEST

Get rid of the key, since we don't need it anymore. The key is anyways stored in an attested CVM (so no intruder can enter), and stored in an encrypted disk.

In [ ]:
! rm -rf dataset.key